# CLIFFGUARD on Google Colab

**TL;DR.**
- This notebook runs real Fold A calibration and Fold B cliff measurement on a Colab GPU — no local hardware required.
- Free T4 (16 GB) handles 1B–3B models in FP16 / NF4; Colab Pro A100 handles up to 8B FP16 and 70B GGUF.
- Every (model, scheme) pair is checkpointed to Google Drive after it finishes, so a disconnected session loses at most one scheme of compute.

**What you'll get out of this notebook.**
1. A calibrated refusal direction `r̂` and per-scheme threshold `τ_q` for the model + schemes your GPU can hold (Fold A).
2. Cliff metrics (`Δ_cliff`, `Δ_W-cliff`, `Δ_B-cliff`) across those schemes on the AdvBench + JailbreakBench corpus (Fold B).
3. A run directory in `/content/drive/MyDrive/cliffguard/results/<run_id>/` with all artifacts and a manifest you can download or commit.

> **Warning — Colab sessions disconnect.**
> Free Colab sessions are pre-empted unpredictably; the runtime can vanish after a few hours of activity. This notebook checkpoints to Drive after every `(model, scheme)` pair so you can re-run any cell and the helper will skip work that is already done.

See [`docs/setup_colab.md`](../docs/setup_colab.md) for the full setup guide and the cell-by-cell map.

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
free, total = torch.cuda.mem_get_info() if torch.cuda.is_available() else (0, 0)
print(f"VRAM free: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB")

## What you should see in C1

| Hardware | Typical free VRAM | What it can run |
|---|---|---|
| T4 (free) | ~14–15 GB free of 16 GB | Llama-3.2-1B/3B FP16, 8B NF4 |
| L4 (Pro) | ~22–23 GB free of 24 GB | Llama-3.1-8B FP16, 13B NF4 |
| A100 40 GB (Pro) | ~38 GB free | 13B FP16, 70B NF4 |
| A100 80 GB (Pro+) | ~78 GB free | 30B FP16, 70B full |

If the free VRAM line shows 0 GB, you are running on CPU — change runtime to GPU via *Runtime → Change runtime type*. The full mapping (and what is **not** possible on free Colab) lives in [`docs/setup_colab.md`](../docs/setup_colab.md#what-fits-on-what).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cliffguard/results
!mkdir -p /content/drive/MyDrive/cliffguard/models
!mkdir -p /content/drive/MyDrive/cliffguard/datasets

## Drive layout

`/content/drive/MyDrive/cliffguard/` is the persistent staging area. The notebook treats it as the source of truth — anything written under `/content/CLIFFGUARD/artifacts/` is copied here after every scheme.

```
/content/drive/MyDrive/cliffguard/
├── datasets/      ← Fold A + Fold B JSONL (linked into data/ in step C14)
├── models/        ← (optional) HF cache redirect; manually move if useful
└── results/
    └── <run_id>/  ← one directory per run; survives session disconnects
        ├── run_metadata.json
        ├── fold_a/   (calibration_summary.json, r_hat_*.npz, checkpoint.json)
        └── fold_b/   (cliff_results.json, checkpoint.json)
```

In [ ]:
%cd /content
# Replace <owner> with your GitHub username or org before running.
!git clone https://github.com/<owner>/CLIFFGUARD.git || (cd CLIFFGUARD && git pull)
%cd /content/CLIFFGUARD
!git log -1 --oneline

> **Edit before running.** Replace `<owner>` in the cell above with the GitHub user or organisation that hosts the repo (e.g. `parnish007`).

In [ ]:
# Preferred: uv-managed venv. If the `!. .venv-colab/bin/activate && uv` pattern
# is flaky on your Colab runtime, fall back to the plain `pip` block below.
!pip install -q uv
!uv venv --python 3.11 .venv-colab
!. .venv-colab/bin/activate && uv pip install -e .
!. .venv-colab/bin/activate && uv pip install -e ".[gpu]"
!. .venv-colab/bin/activate && uv pip install datasets

> **Install fallback.** If the venv activation pattern fails (some Colab runtimes don't inherit `PATH` into `!` shells), use this plain-pip block instead. It installs into Colab's system Python and works everywhere, at the cost of polluting the runtime:
> ```
> !pip install -q -e .
> !pip install -q torch transformers bitsandbytes accelerate
> !pip install -q datasets sentencepiece protobuf
> ```

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install --force-reinstall --no-cache-dir llama-cpp-python -q

## HuggingFace authentication

Meta's Llama models are gated — you must accept each model's license once on its HF page, then paste your **read** access token below. Tokens live at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

Models you will need to accept the license for before running this notebook:
- `meta-llama/Llama-3.2-1B-Instruct`
- `meta-llama/Llama-3.2-3B-Instruct`
- `meta-llama/Llama-Guard-3-8B` (used by the LOOKOUT-JG real judge)
- `mistralai/Mistral-7B-Instruct-v0.3` (alternative rubric grader)

In [ ]:
from huggingface_hub import login
login()  # interactive token prompt

In [ ]:
import sys
sys.path.insert(0, '/content/CLIFFGUARD/notebooks')
import colab_helper as ch
ch.banner()

## Smoke test (no GPU, no datasets)

`scripts/dry_run.py` exercises the full pipeline shape using deterministic synthetic arrays. It returns in under one second and asserts every gate produces a verdict and the CONDUCTOR aggregates to a single BLOCK/ALLOW decision. If this fails, stop here and check the install.

In [ ]:
!python scripts/dry_run.py --tier A --scheme FP16
!python scripts/dry_run.py --tier C --scheme GGUF_Q3_K_M

## Fold A dataset download

First-run download is ~5–10 min for the full corpus (`--max 500` caps it for faster iteration). After the first run, `ch.symlink_datasets_from_drive()` points `data/` at the Drive copy, so subsequent sessions skip the download entirely.

In [ ]:
ch.symlink_datasets_from_drive()
!python scripts/download_fold_a.py --download --max 500

## Configure the run

`ch.choose_model()` reads free VRAM and picks a model + scheme set that fits, using the rule of thumb of ~2 GB per 1B params in FP16 and ~0.5 GB per 1B in NF4:

| Free VRAM | Model | Schemes |
|---|---|---|
| ≥ 35 GB | `meta-llama/Llama-3.1-8B-Instruct` | `[FP16, NF4, AWQ_INT4]` |
| ≥ 14 GB | `meta-llama/Llama-3.2-3B-Instruct` | `[FP16, NF4]` |
| ≥ 8 GB | `meta-llama/Llama-3.2-1B-Instruct` | `[FP16, NF4]` |
| < 8 GB | `Qwen/Qwen2.5-0.5B-Instruct` | `[FP16]` (degraded) |

Override by editing `config['model_id']` and `config['schemes']` before running the next cell.

In [ ]:
config = ch.choose_model()
print(config)

## Run Fold A — calibration with checkpointing

`run_fold_a_with_checkpoint` performs the Arditi difference-in-means refusal-direction calibration for each scheme in `config['schemes']`. After every scheme it:
1. Writes the refusal direction to `artifacts/runs/<run_id>/fold_a/r_hat_<model>_<scheme>.npz`.
2. Updates `fold_a/checkpoint.json` with the scheme as completed.
3. Calls `sync_artifacts_to_drive()` to copy the new files to Drive.

If the cell is re-run after a disconnect, the helper picks up the same `run_id` (matched by `model_id`) and skips any scheme listed in `completed_schemes`.

In [ ]:
ch.run_fold_a_with_checkpoint(config)

## Sync artifacts to Drive

The helper already calls `sync_artifacts_to_drive` after each scheme. Run the cell below if you want to force one extra sync — useful before disconnecting or when you have edited files manually.

In [ ]:
ch.sync_artifacts_to_drive()

## Assemble the Fold B corpus

Fold B uses the AdvBench harmful-behaviour set and JailbreakBench JBB-Behaviors as its adversarial corpus. `ch.assemble_fold_b()` writes the two JSONL files into `data/folds/fold_b/` using the same shape as the rest of the fold loaders. It is idempotent — skips downloads if the files already exist.

In [ ]:
ch.assemble_fold_b()

## Run Fold B — cliff measurement

Fold B reloads the Fold A directions from the run directory, then for each scheme computes:
- `Δ_cliff(q, FP16)` — geometric distance between refusal directions.
- `Δ_W-cliff(q, FP16)` — Wasserstein distance between margin distributions.
- `Δ_B-cliff(q, FP16)` — change in attack-success rate across the threshold.

The result is written to `fold_b/cliff_results.json` along with the `cliff_boundary` (first scheme where all three metrics jump above κ = 0.25 — the H1 verdict for this model family).

In [ ]:
ch.run_fold_b_with_checkpoint(config)
ch.sync_artifacts_to_drive()

## Wrap-up

**Where results live.** `/content/drive/MyDrive/cliffguard/results/<run_id>/` survives the runtime disconnect. To download a copy locally:
1. *Files panel → /content/drive/MyDrive/cliffguard/results* → right-click the run directory → *Download*.
2. Or, from a terminal cell: `!zip -r /content/run.zip /content/drive/MyDrive/cliffguard/results/<run_id> && cp /content/run.zip /content/drive/MyDrive/`.

**Resuming a killed session.**
1. Re-open this notebook and reconnect to a runtime.
2. Re-run cells C1 through C10 to remount Drive, re-clone the repo, and re-install dependencies.
3. Run cell C18 again — `run_fold_a_with_checkpoint` reads the checkpoint in Drive and skips schemes already marked completed.
4. Run cell C24 — `run_fold_b_with_checkpoint` does the same for Fold B.

You lose **at most one scheme** of compute (the one that was running when the runtime died). Everything else is restored from Drive.

**Next steps.** See [`docs/setup_colab.md`](../docs/setup_colab.md) for the full troubleshooting guide and [`docs/cliffguard_complete_guide.md`](../docs/cliffguard_complete_guide.md) for the conceptual reference.